In [1]:
!pip install -q langchain langchain-groq

import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)

In [2]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! Yes, we offer egg‑less versions of all our cakes—plum, chocolate, and cupcakes. Let me know which you’d like, and I’ll give you the price!
Your name is Ravi.


In [3]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()      # create tests.csv first with the %%writefile cell from topic 8

PASS | How much is a plum cake? → The plum cake costs Rs 500. Let me know if you’d like to order!
PASS | Do you make eggless cakes? → Yes, all our cakes—including plum, chocolate, and cupcakes—are availab
FAIL | Can I get a cake today evening? → Sure! Which cake would you like? (Plum Rs 500, Chocolate Rs 650, Cupca
PASS | Do you deliver to Kakinada? → I’ll check with the owner and let you know about delivery to Kakinada.
PASS | What is the price of 6 cupcakes? → 6 cupcakes cost Rs 480. Let me know if you’d like to place an order!

4/5 passed (80%)


In [4]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result["messages"][-1].content)

Yes! All our cakes—plum, chocolate, cupcakes—are available in eggless versions. Let me know which you'd like, and I’ll calculate the total for you.


In [ ]:
# Turning the logic into my business requirement

In [13]:
# 3. Black & White single-sided printing costs Rs. 1 per page.

# No home delivery.
SYSTEM = """
ROLE:
You are a friendly and professional AI assistant
for PrintSmart Printing Shop.

FACTS:
1. We provide document printing services in A4, A3 and A5 sizes.
2. We support PDF files only.
4. Black & White double-sided printing costs Rs. 2 per sheet.
5. Color printing costs Rs. 10 per page.

RULES:
1. Orders must be confirmed by the customer before printing.
2. Customers must collect prints from the shop.
   Never promise an exact completion time without confirmation.
3. Never invent prices, services or order status.
   For unavailable information, say:
   "I'll check with the shop owner."

FORMAT:
- Keep answers under 50 words.
- Be polite and professional.
- Respond in English or Telugu according to the customer.
- Keep answers simple and clear.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

In [14]:
import csv

def run_tests(path: str = "/content/tests.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

FAIL | How much does it cost to print 20 pages in black and white, single-sided? → I’ll check with the shop owner.
PASS | What is the price for 30 sheets printed black and white on both sides? → The cost would be Rs. 60. Please confirm your order, and you can colle
PASS | How much will 10 color pages cost? → Rs. 100 for 10 color pages.
PASS | Can you print an A3 PDF? → Sure! We can print your A3 PDF. Please upload the file and confirm the
PASS | Do you support A5-size PDF printing? → Yes, we support A5‑size PDF printing. Let me know the file and any pre
PASS | Can I send a DOCX file for printing? → We only accept PDF files for printing. Please convert your DOCX to PDF
FAIL | Do you provide home delivery? → I’ll check with the shop owner.
FAIL | How long will my print order take? → I’ll need to confirm your order details first before giving a time est
PASS | Will you start printing immediately after I send the file? → I’ll start printing only after we confirm your order details. Please 

In [ ]:
# Changing and updating the rules and facts again and running the tests again

In [15]:

SYSTEM = """
ROLE:
You are a friendly and professional AI assistant
for PrintSmart Printing Shop.

FACTS:
1. We provide document printing services in A4, A3 and A5 sizes.
2. We support PDF files only.
3. Black & White single-sided printing costs Rs. 1 per page.
4. Black & White double-sided printing costs Rs. 2 per sheet.
5. Color printing costs Rs. 10 per page.

RULES:
1. Orders must be confirmed by the customer before printing.
2. No home delivery. Customers must collect prints from the shop.
   Never promise an exact completion time without confirmation.
3. Never invent prices, services or order status.
   For unavailable information, say:
   "I'll check with the shop owner."

FORMAT:
- Keep answers under 50 words.
- Be polite and professional.
- Respond in English or Telugu according to the customer.
- Keep answers simple and clear.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

In [16]:
run_tests()

PASS | How much does it cost to print 20 pages in black and white, single-sided? → Printing 20 black‑and‑white single‑sided pages costs Rs. 20.  
Please 
PASS | What is the price for 30 sheets printed black and white on both sides? → The cost for 30 double‑sided black & white sheets is Rs. 60.  
Please 
PASS | How much will 10 color pages cost? → Rs. 100 for 10 color pages.  
Please confirm your order before printin
PASS | Can you print an A3 PDF? → Sure, we can print your A3 PDF. Please let me know the number of pages
PASS | Do you support A5-size PDF printing? → Yes, we support A5‑size PDF printing. Please send the PDF file and let
PASS | Can I send a DOCX file for printing? → I’m sorry, but we only accept PDF files for printing. If you can conve
FAIL | Do you provide home delivery? → No, we don’t offer home delivery. Please collect your prints from the 
FAIL | How long will my print order take? → I’ll need to confirm your order details first. Once you let me know th
PASS | Will you 

In [ ]:
# so the observation the pass rate increases as we update the rules and facts

In [ ]:
# Changing the temperature and again running the tests twice to observe the variation

In [17]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

In [18]:
chain = prompt | llm | StrOutputParser()

In [19]:
print("RUN 1")
run_tests()

print("\nRUN 2")
run_tests()

RUN 1
PASS | How much does it cost to print 20 pages in black and white, single-sided? → The cost will be Rs 20 (20 pages × Rs 1 per page).
PASS | What is the price for 30 sheets printed black and white on both sides? → The cost would be Rs. 60 (30 sheets × Rs. 2 per sheet for black‑white 
PASS | How much will 10 color pages cost? → The cost for 10 color pages is Rs. 100 (10 pages × Rs. 10 per page).
PASS | Can you print an A3 PDF? → Yes, we can print your A3 PDF. Please send the PDF file and let us kno
PASS | Do you support A5-size PDF printing? → Yes, we support A5‑size PDF printing. Let me know the details (pages, 
PASS | Can I send a DOCX file for printing? → I’m sorry, but we only accept PDF files for printing. Please convert y
FAIL | Do you provide home delivery? → I’m sorry, but we don’t offer home delivery. Please pick up your print
PASS | How long will my print order take? → I’ll check with the shop owner.
PASS | Will you start printing immediately after I send the file? → We’